# 03 — Data Preparation (CRISP-DM)

Materializa a **base perfeita** para modelagem, com três garantias:

1. **Fail-closed**: a base só avança se passar na validação de esquema/regras.
2. **Anti-leakage**: colunas agregadas do próprio dia-alvo são explicitamente excluídas.
3. **Split temporal estrito**: treino vê o passado; teste cobre o futuro (Q4).

Tudo em código testável (`src/data/preparar_dataset.py`), nunca em célula solta.


## 1) Validação fail-closed da base bruta

In [ ]:
import json

from src.data import preparar_dataset as prep

bruto = prep.carregar_dados_brutos()
problemas = prep.validar_esquema(bruto)
assert problemas == [], problemas
print(f"Base bruta validada: {bruto.shape[0]:,} linhas x {bruto.shape[1]} colunas")

## 2) Limpeza + engenharia de features

In [ ]:
limpo = prep.limpar_dados(bruto)
enriquecido = prep.engenharia_features(limpo)
list(enriquecido.columns)

## 3) Guardrail de vazamento

`quantidade_vendida_mes` agrega o próprio dia-alvo. Ela permanece na base para
relatórios, mas é **excluída** das features.

In [ ]:
print("Colunas com vazamento:", prep.colunas_com_vazamento())
features = prep.selecionar_features(enriquecido)
print(f"Nº de features leakage-safe: {len(features)}")
assert "quantidade_vendida_mes" not in features

## 4) Split temporal estrito

In [ ]:
preparado = prep.marcar_split_temporal(enriquecido)
preparado["split"].value_counts()

In [ ]:
treino_max = preparado.loc[preparado["split"] == "treino", "data"].max()
teste_min = preparado.loc[preparado["split"] == "teste", "data"].min()
print("Último dia do treino:", treino_max.date())
print("Primeiro dia do teste:", teste_min.date())
assert treino_max < teste_min, "Vazamento temporal: futuro dentro do treino."


## 5) Resumo de qualidade e persistência (governança via DVC)

In [ ]:
print(json.dumps(prep.resumo_qualidade(preparado), ensure_ascii=False, indent=2, default=str))

In [ ]:
destino = prep.salvar_parquet(preparado)
print("Base preparada salva em:", destino)

## Conclusão

A base preparada (`dataset_cafeterias_rj_preparado.parquet`) é o **insumo canônico**
das fases seguintes: contém as features leakage-safe, os alvos de negócio e a coluna
`split` que garante a avaliação honesta no futuro. É governada pelo DVC, nunca pelo Git.
